# Neural Networks Model

In [267]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
torch.manual_seed(1234)

import numpy as np

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

import matplotlib.pylab as plt
%matplotlib inline

from data_cleaning import cleaned_data

In [268]:
class Dataset:
    """
    Prepare the dataset into training, validation, and test sets.
    """
    def __init__(self, random_state = 123):
        self.data = cleaned_data()
        y_df = self.data["VOTED"].map({"voted": 1, "not_voted": 0}).to_numpy()

        categorical_feature = ["SEX", "RACE", "EDUC", "EMPSTAT", "NATIVITY", "REGION",
                               "METRO", "MARST", "DIFFMOB"]
        numerical_feature = ["AGE", "FAMSIZE", "NCHILD", "FAMINC", "INCOME_PER_PERSON"]

        ## training set 70%, validation set 10%, test set 20%
        train_x, rest_x, self.train_y, rest_y = train_test_split(
            self.data, y_df, test_size=0.3, random_state=random_state)
        val_x, test_x, self.val_y, self.test_y = train_test_split(
            rest_x, rest_y, test_size=(2/3), random_state=random_state)
        
        self.ohe = OneHotEncoder(sparse_output=False)
        self.scaler = StandardScaler()

        train_x_categorical = self.ohe.fit_transform(train_x[categorical_feature])
        train_x_numerical = self.scaler.fit_transform(train_x[numerical_feature])
        val_x_categorical = self.ohe.transform(val_x[categorical_feature])
        val_x_numerical = self.scaler.transform(val_x[numerical_feature])
        test_x_categorical = self.ohe.transform(test_x[categorical_feature])
        test_x_numerical = self.scaler.transform(test_x[numerical_feature])

        self.train_x = np.concatenate([train_x_categorical, train_x_numerical], axis=1)
        self.val_x = np.concatenate([val_x_categorical, val_x_numerical], axis=1)
        self.test_x = np.concatenate([test_x_categorical, test_x_numerical], axis=1)

        categorical_feature_name = self.ohe.get_feature_names_out(categorical_feature)
        numerical_feature_name = self.scaler.get_feature_names_out(numerical_feature)
        self.feature_order = np.concatenate([categorical_feature_name, numerical_feature_name])

In [269]:
dataset_handler = Dataset()

print("\n"*2, dataset_handler.train_x.shape, "\n"*2)
print(dataset_handler.feature_order, "\n"*2)
print(dataset_handler.train_x[:3])



 (43628, 31) 


['SEX_female' 'SEX_male' 'RACE_asian' 'RACE_black'
 'RACE_indian_aleut_eskimo' 'RACE_others' 'RACE_white' 'EDUC_college_grad'
 'EDUC_hs_grad' 'EDUC_master_higher' 'EMPSTAT_employed'
 'EMPSTAT_not_in_labor_force' 'EMPSTAT_retired' 'EMPSTAT_unemployed'
 'NATIVITY_foreign_born' 'NATIVITY_native_born' 'REGION_midwest'
 'REGION_northeast' 'REGION_south' 'REGION_west' 'METRO_metropolitan'
 'METRO_not_metropolitan' 'MARST_has_spouse' 'MARST_no_spouse'
 'DIFFMOB_mobility_limitation' 'DIFFMOB_no_mobility_limitation' 'AGE'
 'FAMSIZE' 'NCHILD' 'FAMINC' 'INCOME_PER_PERSON'] 


[[ 0.          1.          0.          0.          0.          0.
   1.          1.          0.          0.          0.          1.
   0.          0.          0.          1.          0.          0.
   1.          0.          1.          0.          0.          1.
   0.          1.         -1.73995742 -0.44369352 -0.58541483 -0.41606078
  -0.20372744]
 [ 0.          1.          0.          0.          0.    

In [270]:
X_train = torch.from_numpy(dataset_handler.train_x).float()
y_train = torch.from_numpy(dataset_handler.train_y).float().view(-1, 1)

X_val = torch.from_numpy(dataset_handler.val_x).float()
y_val = torch.from_numpy(dataset_handler.val_y).float().view(-1, 1)

X_test = torch.from_numpy(dataset_handler.test_x).float()
y_test = torch.from_numpy(dataset_handler.test_y).float().view(-1, 1)

In [271]:
## Pure SDG took too much time to train the model. Due to this problem, mini-batch was introduced.
train_data = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_data, batch_size=50, shuffle=True)

In [272]:
class TorchNetwork(nn.Module):

    def __init__(self, num_features) -> None:
        super().__init__()
        self.n = nn.Sequential(nn.Linear(num_features, 31), nn.ReLU(),
                               nn.Linear(31, 31), nn.ReLU(),
                               nn.Linear(31, 31), nn.ReLU(),
                               nn.Linear(31, 1))

    def forward(self, x):
        return self.n(x)


In [273]:
model = TorchNetwork(dataset_handler.train_x.shape[1])

In [281]:
optim = torch.optim.SGD(model.parameters(), lr=0.003, weight_decay=0.0001)
pos_weight = torch.tensor([(y_train == 0).sum()/(y_train == 1).sum()])
loss_func = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

In [282]:
for epoch in range(50):
    model.train()
    total_loss = 0

    for x, y in train_loader:

        optim.zero_grad()

        output = model(x)
        loss = loss_func(output, y)

        loss.backward()
        optim.step()
        total_loss += loss.item() * x.size(0)
    
    model.eval()
    with torch.no_grad():
        result = model(X_val)
        hat_y = (torch.sigmoid(result) > 0.5).float()
        f1 = f1_score(y_val.numpy(), hat_y.numpy())

    print(f"Epoch {epoch+1}, Loss: {total_loss/len(X_train):.4f}, F1: {f1:.4f}")


Epoch 1, Loss: 0.2651, F1: 0.7597
Epoch 2, Loss: 0.2651, F1: 0.7673
Epoch 3, Loss: 0.2650, F1: 0.7759
Epoch 4, Loss: 0.2650, F1: 0.7758
Epoch 5, Loss: 0.2650, F1: 0.7624
Epoch 6, Loss: 0.2650, F1: 0.7688
Epoch 7, Loss: 0.2650, F1: 0.7697
Epoch 8, Loss: 0.2650, F1: 0.7660
Epoch 9, Loss: 0.2650, F1: 0.7666
Epoch 10, Loss: 0.2650, F1: 0.7672
Epoch 11, Loss: 0.2650, F1: 0.7712
Epoch 12, Loss: 0.2649, F1: 0.7719
Epoch 13, Loss: 0.2650, F1: 0.7673
Epoch 14, Loss: 0.2649, F1: 0.7617
Epoch 15, Loss: 0.2649, F1: 0.7696
Epoch 16, Loss: 0.2649, F1: 0.7689
Epoch 17, Loss: 0.2649, F1: 0.7608
Epoch 18, Loss: 0.2649, F1: 0.7691
Epoch 19, Loss: 0.2649, F1: 0.7674
Epoch 20, Loss: 0.2649, F1: 0.7715
Epoch 21, Loss: 0.2649, F1: 0.7645
Epoch 22, Loss: 0.2649, F1: 0.7674
Epoch 23, Loss: 0.2648, F1: 0.7619
Epoch 24, Loss: 0.2648, F1: 0.7696
Epoch 25, Loss: 0.2648, F1: 0.7703
Epoch 26, Loss: 0.2648, F1: 0.7688
Epoch 27, Loss: 0.2648, F1: 0.7680
Epoch 28, Loss: 0.2648, F1: 0.7589
Epoch 29, Loss: 0.2648, F1: 0

In [276]:

model.eval()
with torch.no_grad():
    result = model(X_test)
    hat_y = (torch.sigmoid(result) > 0.5).float()
    f1 = f1_score(y_test.numpy(), hat_y.numpy())

print("F1:", f1)


F1: 0.7574611181168558
